# Lesson 2: Autograd and many features

Goes with the Lesson 2 page of the Forkcast course. PyTorch computes the slopes; the model grows from one feature to four.

Runtime: CPU is enough.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. Autograd on Lesson 1's data

Same guess as last lesson, `w = 1, b = 2`, so the numbers are comparable.

In [ ]:
import torch
from forkcast.data import make_deliveries
from forkcast import linear
x, y = make_deliveries(200, seed=0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(2.0, requires_grad=True)
print("before backward:", w.grad)
loss = ((w * x + b - y) ** 2).mean()
print(loss)
loss.backward()
print("autograd:", w.grad, b.grad)
print("lesson 1:", linear.gradients(x, y, w.detach(), b.detach()))

### Gradients add up

Call `backward()` three times without zeroing. The data and the loss never change; only `.grad` does.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    ((w * x + b - y) ** 2).mean().backward()
    print(i + 1, w.grad)
w.grad.zero_()
print("after zero_():", w.grad)

## 3. Write the three functions

Run the cell to write `forkcast/regression.py`. Build them in order: `autograd_gradients` (step 3 on the page), `predict_many` (step 5), `fit_autograd` (step 6). Re-run after every change.

In [ ]:
%%writefile forkcast/regression.py
"""Lesson 2. Let PyTorch find the slopes, and go from one feature to many.

``X`` has shape (N, d): one row per order, one column per feature.
``w`` has shape (d,): one weight per feature. ``b`` is a single number, shape ().
Replace every ``raise NotImplementedError`` with a few lines of code.
"""

from __future__ import annotations

import torch
from torch import Tensor


def autograd_gradients(x: Tensor, y: Tensor, w: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    """Lesson 1's ``gradients``, but computed by autograd instead of by hand.

    ``w`` and ``b`` are 0-d tensors created with ``requires_grad=True``.
    Compute the mean squared error of ``w * x + b`` against ``y``, call
    ``.backward()`` on it, and return copies of ``w.grad`` and ``b.grad``.
    """
    raise NotImplementedError("Lesson 2, step 3: loss = ...; loss.backward(); return w.grad.clone(), b.grad.clone()")


def predict_many(X: Tensor, w: Tensor, b: Tensor) -> Tensor:
    """A weighted sum of every feature, for every order at once.

    (N, d) @ (d,) + () -> (N,)
    """
    raise NotImplementedError("Lesson 2, step 5: X @ w + b")


def fit_autograd(X: Tensor, y: Tensor, lr: float = 0.005, steps: int = 4000):
    """Gradient descent on w (shape (d,)) and b (shape ()), with autograd doing the slopes.

    Start w at zeros and b at zero, both with ``requires_grad=True``. Each step:
    predict, compute the mean squared error, record ``loss.item()``, call
    ``loss.backward()``, update w and b inside ``torch.no_grad()``, then zero
    both ``.grad`` tensors. Return ``(w.detach(), b.detach(), losses)``.
    """
    raise NotImplementedError("Lesson 2, step 6: the training loop with autograd")

## 4. Four features

In [ ]:
import importlib
from forkcast import regression
importlib.reload(regression)
from forkcast.data import make_orders, FEATURES, TRUE_WEIGHTS, TRUE_BIAS

X, y = make_orders(400, seed=0)
print(X.shape, y.shape)
print(FEATURES)
print(X[:3])
print(y[:3])

w4 = torch.tensor([3.5, 1.5, 4.0, 6.0]); b4 = torch.tensor(8.0)
pred = regression.predict_many(X, w4, b4)
print(pred.shape)
print(torch.allclose(pred, (X * w4).sum(dim=1) + b4))     # @ is the loop you would have written

## 5. Fit, and read what it learned

In [ ]:
importlib.reload(regression)
w, b, losses = regression.fit_autograd(X, y, lr=0.005, steps=4000)
for name, learned, true in zip(FEATURES, w.tolist(), TRUE_WEIGHTS.tolist()):
    print(f"{name:12s} learned {learned:5.2f}   truth {true}")
print(f"bias         learned {b.item():5.2f}   truth {TRUE_BIAS}")
print(f"loss went from {losses[0]:.1f} to {losses[-1]:.2f}")

import matplotlib.pyplot as plt
plt.plot(losses); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss (log scale)");

### Break it on purpose

The same loop with the two `zero_()` lines deleted. Nothing crashes; the loss just stops making sense.

In [ ]:
w_ = torch.zeros(4, requires_grad=True); b_ = torch.zeros((), requires_grad=True)
bad = []
for step in range(30):
    loss = ((X @ w_ + b_ - y) ** 2).mean()
    bad.append(round(loss.item(), 1))
    loss.backward()
    with torch.no_grad():
        w_ -= 0.005 * w_.grad
        b_ -= 0.005 * b_.grad
    # no zero_() here, on purpose
print(bad)

## 6. Definition of done

Expected: `6 passed`, then `20 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_02.py -q

In [ ]:
!pytest -m fast -q

## 7. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 2: autograd and many features"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main